### First, we turn each individual EHR into it's own .txt to ensure that the windows LIWC uses only contain text from that note (not surrounding notes if the target word occurs near the end of the EHR)

In [ ]:
import os
from tqdm import tqdm

input_folder = "input_dir"
output_folder = "output_dir"
delimiter = "---END OF EHR---"

In [ ]:
os.makedirs(output_folder, exist_ok=True)

In [ ]:
for filename in tqdm(os.listdir(input_folder), desc="Docs split:"):
    if filename.endswith(".txt"):
        input_path = os.path.join(input_folder, filename)

        with open(input_path, "r", encoding="utf-8") as f:
            content = f.read()

        parts = [p.strip() for p in content.split(delimiter) if p.strip()]

        base_name = os.path.splitext(filename)[0]
        for i, part in enumerate(parts):
            output_path = os.path.join(output_folder, f"{base_name}_{i}.txt")
            with open(output_path, "w", encoding="utf-8") as out:
                      out.write(part)

        print(f"Split {filename} into {len(parts)} parts.")

### Next, use LIWC's Contextualizer feature to extract the +/- 10 token snippets
- Select the split folder (output folder above) as the folder to analyze
- Set both windows to 10
- Load External Dictionary (current draft. make sure to have at least one column for which EVERY term is checked. Select this column as the category to analyze). This requires using the Dictionary Workbench to turn an .xlsx or .csv file into a .dicx file, saving the .dicx file, and loading that. No need to omit punctuation.
- Save the output .csv
- In excel, replace all '=' and '-' with ' ' to ensure no cells are being treated as formulas

In [ ]:
import pandas as pd
import re

liwc_context = r"contextualizer_output_file_location"
context_df = pd.read_csv(liwc_context, encoding='latin1', header='infer')
context_df = context_df.dropna(axis='columns', how='all')
context_df = context_df.dropna(axis='rows', how='any')
context_df['Context'] = context_df['Context Left'] + " " + context_df['Match'] + " " + context_df['Context Right']
context_df

In [ ]:
stopwords = ['and', 'of', 'to', 'the', 'with', 'is', 'was', 'are', 'does', 'for', 'in', 'any',
             'has', 'had', 'have', 'this', 'that', 'on', 'in',
             'on', 'with', 'as', 'by', 'a', 'an', 'from', 'it', 'been']

In [ ]:
from collections import Counter
pattern = re.compile(r'\b[a-zA-Z][a-zA-Z]+\b')

def top_co_words(group_df):
    snippets = []
    for snippet in group_df['Context']:
        words = list(pattern.findall(snippet.lower()))
        words = [w for w in words if w not in stopwords]
        snippets.extend(words)
    occ = dict(Counter(snippets).items())
    sorted_by_value = sorted(occ.items(), key=lambda item: item[1], reverse=True)
    top_co = dict(sorted_by_value[:10])
    return(top_co)

top_co = context_df.groupby('Match').apply(top_co_words, include_groups=False)

In [ ]:
top_co_df = pd.DataFrame(top_co)
top_co_df

In [ ]:
top_co.to_csv(r"output_file_location")